# Synthetic Data Contamination Research Pipeline (Colab Runner)

This notebook executes the pipeline on a remote GPU (e.g. Google Colab T4).
Follow the cells step by step.

### Step 1: Check GPU presence
A CUDA GPU is required for running HuggingFace generators and feature extraction.

In [ ]:
# 1. Verify GPU availability
import subprocess
import sys

try:
    res = subprocess.run(["nvidia-smi"], capture_output=True, text=True, check=True)
    print(res.stdout)
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA is not available to PyTorch.")
    print(f"CUDA Device: {torch.cuda.get_device_name(0)}")
except Exception as e:
    sys.exit("ERROR: No GPU detected! Please navigate to Runtime -> Change runtime type, select T4 GPU, and restart runtime.")


### Step 2: Clone repository & install dependencies

In [ ]:
# 2. Clone repository & install dependencies
import os

REPO_URL = "https://github.com/ratik12/synthetic-contamination.git"
REPO_NAME = "synthetic-contamination"

if not os.path.exists(REPO_NAME):
    !git clone {REPO_URL}

%cd {REPO_NAME}
!pip install --upgrade pip
!pip install -r requirements.txt


### Step 3: Mount Google Drive & symlink data/ and results/ directories
Persisting data/ and results/ on Google Drive ensures that session timeouts or disconnects do not cause loss of work.

In [ ]:
# 3. Mount Google Drive and create symlinks for persistence
from google.colab import drive
from pathlib import Path
import os

drive.mount("/content/drive")

drive_dir = Path("/content/drive/MyDrive/synthetic_contamination_backup")
(drive_dir / "data").mkdir(parents=True, exist_ok=True)
(drive_dir / "results").mkdir(parents=True, exist_ok=True)

local_data = Path("data")
local_results = Path("results")

if local_data.is_symlink() or local_data.exists():
    if local_data.is_symlink():
        local_data.unlink()
if not local_data.exists():
    os.symlink(str(drive_dir / "data"), "data")

if local_results.is_symlink() or local_results.exists():
    if local_results.is_symlink():
        local_results.unlink()
if not local_results.exists():
    os.symlink(str(drive_dir / "results"), "results")

print("Linked data/ ->", os.path.realpath("data"))
print("Linked results/ ->", os.path.realpath("results"))


### Step 4: Run environment check

In [ ]:
# Run 00_check_env
!python scripts/00_check_env.py --config configs/mid.yaml


### Step 5: Prepare data split manifest

In [ ]:
# Run 01_prepare_data
!python scripts/01_prepare_data.py --config configs/mid.yaml


### Step 6: Smoke test (16 passages per generator) and inspect samples
Inspect the samples carefully to ensure there are no refusals, markdown, or chat preambles.

In [ ]:
# Smoke test generation (limit=16) and inspect raw samples
!python scripts/02_generate.py --config configs/mid.yaml --extra limit=16
!python scripts/show_samples.py --config configs/mid.yaml --raw --n 4


### Step 7: Run full pipeline
Executes all stages (full generation, corpus building, feature extraction, detector training, evaluation, audit estimation, downstream preparation). Resumes automatically from the smoke test output.

In [ ]:
# Run full pipeline
!bash run_all.sh configs/mid.yaml


### Step 8: Generate report tables & figures

In [ ]:
# Generate LaTeX tables and figures
!python scripts/09_make_report_tables.py --config configs/mid.yaml --extra out=report


### Step 9: Print results summary CSVs

In [ ]:
# Print detection metrics and estimator summary
import pandas as pd
from IPython.display import display

print("=== DETECTION METRICS (results/mid_today/detection_metrics.csv) ===")
det_csv = Path("results/mid_today/detection_metrics.csv")
if det_csv.exists():
    display(pd.read_csv(det_csv))
else:
    print(f"File not found: {det_csv}")

print("
=== ESTIMATOR SUMMARY (results/mid_today/estimator_summary.csv) ===")
est_csv = Path("results/mid_today/estimator_summary.csv")
if est_csv.exists():
    display(pd.read_csv(est_csv))
else:
    print(f"File not found: {est_csv}")
